In [ ]:
# ============================================================
# Photo Cuba v2 — Inference Worker
# Run in Google Colab (T4 GPU) OR locally in VS Code.
# After running Cell 5, set in .env:
#   INFERENCE_API_URL=http://localhost:8001   (local)
#   INFERENCE_API_URL=https://xxxx.ngrok-free.app  (Colab)
# ============================================================

# ── CELL 1: Install dependencies ─────────────────────────────
import sys, subprocess, importlib

def _has_cuda():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return False

CUDA = _has_cuda()
print(f"CUDA available: {CUDA}")

if CUDA:
    pkgs = "insightface==0.7.3 onnxruntime-gpu torch torchvision huggingface_hub fastapi uvicorn pyngrok faiss-gpu numpy safetensors"
else:
    pkgs = "insightface==0.7.3 onnxruntime torch torchvision huggingface_hub fastapi uvicorn pyngrok faiss-cpu numpy safetensors"

subprocess.run([sys.executable, "-m", "pip", "install"] + pkgs.split() + ["--quiet"], check=True)
print("Dependencies installed.")


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ── CELL 2: Environment setup ────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/TANTRA 2026'
    os.chdir(WORK_DIR)
    IN_COLAB = True
    print("Running in Colab.")
    print("Working directory:", os.getcwd())
    print("Contents:", os.listdir('.'))
except ModuleNotFoundError:
    # Running locally in VS Code — skip Drive mount entirely
    IN_COLAB = False
    print("Running locally (VS Code). Drive mount skipped.")
    print("Working directory:", os.getcwd())


ModuleNotFoundError: No module named 'google'

In [ ]:
# ── CELL 3: Load models ──────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import cv2
import numpy as np
from insightface.app import FaceAnalysis
import sys, os, urllib.request

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── InsightFace — detection + landmark only ───────────────────
print("Loading InsightFace detector (buffalo_l)...")
detector = FaceAnalysis(
    name="buffalo_l",
    allowed_modules=["detection", "landmark_2d_106"],
)
detector.prepare(ctx_id=0 if DEVICE == "cuda" else -1, det_size=(640, 640))
print("InsightFace detector ready.")

# ── AdaFace IR-50 — recognition backbone ─────────────────────
# Architecture matches the official AdaFace repo exactly.
# Weights: adaface_ir50_webface4m.ckpt from GitHub releases.

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False)

class IBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bn1       = nn.BatchNorm2d(inplanes, eps=1e-05)
        self.conv1     = conv3x3(inplanes, planes)
        self.bn2       = nn.BatchNorm2d(planes, eps=1e-05)
        self.prelu     = nn.PReLU(planes)
        self.conv2     = conv3x3(planes, planes, stride)
        self.bn3       = nn.BatchNorm2d(planes, eps=1e-05)
        self.downsample = downsample
        self.stride    = stride

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out

class IResNet(nn.Module):
    def __init__(self, layers, dropout=0.0, num_features=512):
        super().__init__()
        self.inplanes = 64
        self.conv1    = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(64, eps=1e-05)
        self.prelu    = nn.PReLU(64)
        self.layer1   = self._make_layer(64,  layers[0], stride=2)
        self.layer2   = self._make_layer(128, layers[1], stride=2)
        self.layer3   = self._make_layer(256, layers[2], stride=2)
        self.layer4   = self._make_layer(512, layers[3], stride=2)
        self.bn2      = nn.BatchNorm2d(512, eps=1e-05)
        self.dropout  = nn.Dropout(p=dropout)
        self.fc       = nn.Linear(512 * 7 * 7, num_features)
        self.features = nn.BatchNorm1d(num_features, eps=1e-05)
        nn.init.constant_(self.features.weight, 1.0)
        self.features.weight.requires_grad_(False)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes, stride),
                nn.BatchNorm2d(planes, eps=1e-05),
            )
        layers = [IBasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, n_blocks):
            layers.append(IBasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.prelu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.dropout(self.bn2(x))
        x = self.fc(x.flatten(1))
        x = self.features(x)
        return F.normalize(x, p=2, dim=-1)

# IResNet50 uses [3, 4, 14, 3] — note layer3 has 14 blocks (not 6 like standard ResNet50)
def iresnet50(dropout=0.0):
    return IResNet([3, 4, 14, 3], dropout=dropout)


USE_ADAFACE = True

if USE_ADAFACE:
    ADAFACE_CKPT = os.path.join(os.path.expanduser("~"), ".photo_cuba_adaface_ir50_webface4m.ckpt")
    ADAFACE_URL  = "https://github.com/mk-minchul/AdaFace/releases/download/webface4m/adaface_ir50_webface4m.ckpt"

    if not os.path.exists(ADAFACE_CKPT):
        print("Downloading AdaFace IR-50 webface4m checkpoint (~170 MB)...")
        urllib.request.urlretrieve(ADAFACE_URL, ADAFACE_CKPT)
        print("Download complete.")
    else:
        print(f"AdaFace checkpoint cached: {ADAFACE_CKPT}")

    adaface_model = iresnet50().to(DEVICE)
    ckpt  = torch.load(ADAFACE_CKPT, map_location=DEVICE, weights_only=False)
    state = ckpt.get("state_dict", ckpt)
    # Strip DataParallel 'module.' prefix if present
    state = {(k[7:] if k.startswith("module.") else k): v for k, v in state.items()}
    missing, unexpected = adaface_model.load_state_dict(state, strict=False)
    adaface_model.eval()
    print(f"AdaFace IR-50 ready | device={DEVICE} | missing={len(missing)} unexpected={len(unexpected)}")
    if missing:
        print(f"  Missing keys (first 5): {missing[:5]}")
else:
    adaface_model = None
    print("Using buffalo_l ArcFace recognition (AdaFace disabled).")


In [ ]:
# ── CELL 4: Define embedding extraction ──────────────────────

ADAFACE_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

DET_SCORE_THRESHOLD = 0.60
QUALITY_THRESHOLD   = 0.15   # Laplacian variance normalized


def blur_score(crop_bgr: np.ndarray) -> float:
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    return min(cv2.Laplacian(gray, cv2.CV_64F).var() / 500.0, 1.0)


def get_embeddings(img_bgr: np.ndarray, augment: bool = False) -> dict:
    """
    Full pipeline: detect → align → quality gate → embed.

    augment=True for query images (flip + brightness).
    augment=False for dataset indexing (speed priority).

    Returns:
        {
          "embeddings": [[512 floats], ...],
          "det_scores": [float, ...],
          "count": int
        }
    """
    results = {"embeddings": [], "det_scores": [], "count": 0}

    images_to_process = [img_bgr]
    if augment:
        images_to_process.append(cv2.flip(img_bgr, 1))
        images_to_process.append(
            cv2.convertScaleAbs(img_bgr, alpha=1.2, beta=0)
        )

    all_embs = []

    for img in images_to_process:
        faces = detector.get(img)
        for f in faces:
            if f.det_score < DET_SCORE_THRESHOLD:
                continue

            # Aligned 112x112 crop
            from insightface.utils import face_align
            crop = face_align.norm_crop(img, landmark=f.kps)

            if blur_score(crop) < QUALITY_THRESHOLD:
                continue

            if USE_ADAFACE and adaface_model is not None:
                rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                tensor = ADAFACE_TRANSFORM(rgb).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    emb = adaface_model(tensor)
                emb = emb.cpu().numpy()[0]
            else:
                # buffalo_l ArcFace fallback
                emb = f.embedding
                norm = np.linalg.norm(emb)
                if norm < 1e-6:
                    continue
                emb = emb / norm

            all_embs.append(emb.astype("float32"))
            results["det_scores"].append(float(f.det_score))

    if augment and len(all_embs) > 1:
        # Mean-pool across augmentations → stable query centroid
        mean_emb = np.mean(all_embs, axis=0)
        mean_emb = mean_emb / np.linalg.norm(mean_emb)
        results["embeddings"] = [mean_emb.tolist()]
    else:
        results["embeddings"] = [e.tolist() for e in all_embs]

    results["count"] = len(results["embeddings"])
    return results


In [ ]:
# ── CELL 5: FastAPI inference server ─────────────────────────

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import uvicorn, threading, io
from PIL import Image

inference_app = FastAPI(title="Photo Cuba Colab Inference Worker")


@inference_app.get("/")
def root():
    return {"status": "ok", "device": DEVICE, "adaface": USE_ADAFACE}


@inference_app.post("/embed")
async def embed(
    image: UploadFile = File(...),
    augment: str = "false",
):
    """
    Receive image → return face embeddings.
    Called by laptop API for every index/search operation.
    """
    try:
        img_bytes = await image.read()
        img_array = np.frombuffer(img_bytes, np.uint8)
        img_bgr   = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        if img_bgr is None:
            return JSONResponse(
                {"embeddings": [], "det_scores": [], "count": 0,
                 "error": "could_not_decode_image"},
                status_code=422
            )

        result = get_embeddings(img_bgr, augment=(augment == "true"))
        return result

    except Exception as e:
        return JSONResponse(
            {"embeddings": [], "det_scores": [], "count": 0,
             "error": str(e)},
            status_code=500
        )


def _run_server():
    uvicorn.run(inference_app, host="0.0.0.0", port=8001, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
print("Inference server started on port 8001.")

In [ ]:

# ── CELL 6: Expose via ngrok ──────────────────────────────────
# Copy the URL printed below → paste into .env as INFERENCE_API_URL

from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Free ngrok — no auth token needed for basic use
# For stable URL, create free account at ngrok.com and set:
#   ngrok.set_auth_token("your_token")

tunnel = ngrok.connect(8001)
public_url = tunnel.public_url
print("\n" + "="*50)
print("INFERENCE WORKER URL:")
print(public_url)
print("="*50)
print("\nPaste this into your .env file:")
print(f"INFERENCE_API_URL={public_url}")
print("\nKeep this Colab tab open during your event!")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Robust Multi-Source Indexer (Drive side)
#
# Handles:
#   - Multiple Drive root directories
#   - Nested subfolders at any depth
#   - Images and videos mixed anywhere
#   - Duplicate files across folders (hash-based dedup)
#   - Resume from checkpoint (indexing_log.json on Drive)
#   - Resize guard for DSLR photos (>4MP)
#   - Shared log with HDD indexer (laptop batch_index.py)
# ════════════════════════════════════════════════════════════

import uuid, time, cv2, os, json, hashlib
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# ── Config ────────────────────────────────────────────────────
QDRANT_URL     = "your_qdrant_url_here"
QDRANT_API_KEY = "your_qdrant_api_key_here"
EVENT_ID       = "tantra_2026"
EMBEDDING_DIM  = 512

# ALL Drive roots to scan — add as many as you have
DRIVE_ROOTS = [
    "/content/drive/MyDrive/TANTRA 2026",
    # "/content/drive/MyDrive/TANTRA 2026 Backup",
]

# Shared log — both Colab AND laptop write here for unified resume
LOG_FILE    = "/content/drive/MyDrive/TANTRA 2026/indexing_log.json"
BATCH_SIZE  = 50

IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff", ".tif"}
VIDEO_EXTS  = {".mp4", ".mov", ".avi", ".mkv", ".wmv", ".m4v", ".mts", ".ts"}
VIDEO_FRAME_INTERVAL_SEC = 5
INDEX_VIDEOS = False   # flip to True after images confirmed working


# ════════════════════════════════════════════════════════════
# Scanner
# ════════════════════════════════════════════════════════════

def scan_roots(roots):
    images, videos, scan_skipped = [], [], []
    seen_hashes      = set()
    subfolder_counts = defaultdict(int)

    for root in roots:
        root_path = Path(root)
        if not root_path.exists():
            print(f"  WARNING: Root not found, skipping: {root}")
            continue

        print(f"Scanning: {root}")
        file_count = 0

        for path in sorted(root_path.rglob("*")):
            if not path.is_file():
                continue

            ext = path.suffix.lower()

            if path.name.startswith(".") or path.name.startswith("~"):
                continue
            if ext not in IMAGE_EXTS and ext not in VIDEO_EXTS:
                continue

            try:
                size = path.stat().st_size
            except Exception:
                scan_skipped.append((str(path), "stat failed"))
                continue

            if size < 5000:
                scan_skipped.append((str(path), f"too small {size}B"))
                continue

            # Dedup by 64KB hash + size
            try:
                with open(path, "rb") as f:
                    head = f.read(65536)
                fhash = hashlib.md5(head + str(size).encode()).hexdigest()
                if fhash in seen_hashes:
                    scan_skipped.append((str(path), "duplicate"))
                    continue
                seen_hashes.add(fhash)
            except Exception:
                pass

            try:
                rel = str(path.parent.relative_to(root))
            except ValueError:
                rel = str(path.parent)
            subfolder_counts[f"{Path(root).name}/{rel or '(root)'}"] += 1

            if ext in IMAGE_EXTS:
                images.append(path)
            else:
                videos.append(path)
            file_count += 1

        print(f"  {file_count} unique media files found\n")

    return images, videos, scan_skipped, subfolder_counts


# ════════════════════════════════════════════════════════════
# Log helpers
# ════════════════════════════════════════════════════════════

def load_log():
    if Path(LOG_FILE).exists():
        try:
            with open(LOG_FILE) as f:
                return json.load(f)
        except Exception:
            print("WARNING: Log corrupted, starting fresh")
    return {"indexed": [], "skipped": [], "failed": [], "faces_total": 0}

def save_log(log):
    Path(LOG_FILE).parent.mkdir(parents=True, exist_ok=True)
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)

def get_done_filenames(log) -> set:
    done = set()
    for entry in log["indexed"] + log["skipped"] + log["failed"]:
        if isinstance(entry, dict):
            name = entry.get("filename") or entry.get("file", "")
            if name:
                done.add(Path(name).name)
        elif isinstance(entry, str):
            done.add(Path(entry).name)
    return done


# ════════════════════════════════════════════════════════════
# Index image
# ════════════════════════════════════════════════════════════

def index_image(img_path: Path, root: str, batch: list, log: dict):
    try:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            log["skipped"].append({"filename": img_path.name, "reason": "cv2 read failed", "source": "drive"})
            return "skipped"

        h, w = img_bgr.shape[:2]
        if h * w > 4_000_000:
            scale   = (4_000_000 / (h * w)) ** 0.5
            img_bgr = cv2.resize(img_bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

        result = get_embeddings(img_bgr, augment=False)

        if not result["embeddings"]:
            log["skipped"].append({"filename": img_path.name, "reason": "no face", "source": "drive"})
            return "skipped"

        try:
            rel = str(img_path.parent.relative_to(root))
        except ValueError:
            rel = str(img_path.parent)

        for emb in result["embeddings"]:
            batch.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=emb,
                payload={
                    "image_path":    str(img_path),
                    "image_url":     "",
                    "filename":      img_path.name,
                    "subfolder":     rel or "(root)",
                    "media_type":    "image",
                    "source":        "drive",
                    "det_score":     1.0,
                    "quality_score": 1.0,
                    "event_id":      EVENT_ID,
                    "timestamp":     datetime.now().isoformat(),
                }
            ))
            log["faces_total"] += 1

        log["indexed"].append({
            "filename": img_path.name,
            "path":     str(img_path),
            "subfolder": rel or "(root)",
            "faces":    len(result["embeddings"]),
            "source":   "drive",
        })
        return "indexed"

    except Exception as e:
        log["failed"].append({"filename": img_path.name, "error": str(e), "source": "drive"})
        return "failed"


# ════════════════════════════════════════════════════════════
# Index video
# ════════════════════════════════════════════════════════════

def index_video(vid_path: Path, root: str, batch: list, log: dict):
    try:
        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            log["skipped"].append({"filename": vid_path.name, "reason": "cannot open", "source": "drive"})
            return "skipped"

        fps          = cap.get(cv2.CAP_PROP_FPS) or 25
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_s   = total_frames / fps
        step         = max(1, int(fps * VIDEO_FRAME_INTERVAL_SEC))

        try:
            rel = str(vid_path.parent.relative_to(root))
        except ValueError:
            rel = str(vid_path.parent)

        faces_added = 0
        frame_idx   = 0

        while frame_idx < total_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]
            if h * w > 2_000_000:
                scale = (2_000_000 / (h * w)) ** 0.5
                frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

            result = get_embeddings(frame, augment=False)
            for emb in result["embeddings"]:
                batch.append(PointStruct(
                    id=str(uuid.uuid4()),
                    vector=emb,
                    payload={
                        "image_path":    str(vid_path),
                        "image_url":     "",
                        "filename":      vid_path.name,
                        "subfolder":     rel or "(root)",
                        "media_type":    "video",
                        "frame_sec":     round(frame_idx / fps, 2),
                        "source":        "drive",
                        "det_score":     1.0,
                        "quality_score": 1.0,
                        "event_id":      EVENT_ID,
                        "timestamp":     datetime.now().isoformat(),
                    }
                ))
                faces_added        += 1
                log["faces_total"] += 1

            frame_idx += step

        cap.release()

        log["indexed"].append({
            "filename":    vid_path.name,
            "path":        str(vid_path),
            "subfolder":   rel or "(root)",
            "duration_s":  round(duration_s, 1),
            "faces":       faces_added,
            "source":      "drive",
        })
        return "indexed"

    except Exception as e:
        log["failed"].append({"filename": vid_path.name, "error": str(e), "source": "drive"})
        return "failed"


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
existing = [c.name for c in qdrant.get_collections().collections]
if EVENT_ID not in existing:
    qdrant.create_collection(
        EVENT_ID,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE)
    )
    print(f"Created Qdrant collection: {EVENT_ID}")
else:
    info = qdrant.get_collection(EVENT_ID)
    print(f"Collection exists: {EVENT_ID} ({info.points_count} faces)")

# Scan
images, videos, scan_skipped, subfolder_counts = scan_roots(DRIVE_ROOTS)

print(f"{'─'*55}")
print(f"  Unique images found : {len(images)}")
print(f"  Unique videos found : {len(videos)}")
print(f"  Scan-skipped        : {len(scan_skipped)}")
print(f"{'─'*55}")
print(f"\nFolder breakdown:")
for folder, count in sorted(subfolder_counts.items()):
    print(f"  {folder:52s}: {count}")
print()

# Resume log
log  = load_log()
done = get_done_filenames(log)
print(f"Already done (Drive + HDD combined): {len(done)} files")

todo_images = [p for p in images if p.name not in done]
todo_videos = [p for p in videos if p.name not in done] if INDEX_VIDEOS else []
todo_total  = len(todo_images) + len(todo_videos)

print(f"Images to index : {len(todo_images)}")
if not INDEX_VIDEOS and videos:
    print(f"Videos          : {len(videos)} found — set INDEX_VIDEOS=True to process")
print(f"Total to process: {todo_total}\n")

if todo_total == 0:
    print("Nothing to do — all files already indexed!")
else:
    def find_root(p: Path) -> str:
        for r in DRIVE_ROOTS:
            try:
                p.relative_to(r)
                return r
            except ValueError:
                continue
        return DRIVE_ROOTS[0]

    batch      = []
    counters   = {"indexed": 0, "skipped": 0, "failed": 0}
    start_time = time.time()
    all_todo   = todo_images + todo_videos

    for i, file_path in enumerate(all_todo):
        root    = find_root(file_path)
        ext     = file_path.suffix.lower()
        outcome = index_image(file_path, root, batch, log) if ext in IMAGE_EXTS \
                  else index_video(file_path, root, batch, log)

        counters[outcome] = counters.get(outcome, 0) + 1

        if len(batch) >= BATCH_SIZE:
            qdrant.upsert(collection_name=EVENT_ID, points=batch)
            batch = []

        if (i + 1) % 50 == 0 or (i + 1) == todo_total:
            elapsed   = time.time() - start_time
            rate      = (i + 1) / elapsed
            remaining = (todo_total - i - 1) / rate if rate > 0 else 0
            print(
                f"  [{i+1:4d}/{todo_total}] {(i+1)/todo_total*100:5.1f}%  "
                f"indexed={counters['indexed']}  "
                f"skipped={counters['skipped']}  "
                f"failed={counters['failed']}  "
                f"faces={log['faces_total']}  "
                f"ETA={remaining/60:.1f}min"
            )
            save_log(log)

    if batch:
        qdrant.upsert(collection_name=EVENT_ID, points=batch)
    save_log(log)

    elapsed    = time.time() - start_time
    final_info = qdrant.get_collection(EVENT_ID)

    print(f"""
{'='*55}
  TANTRA 2026 Drive Indexing Complete
{'='*55}
  Files processed : {todo_total}
  Indexed         : {counters['indexed']}
  Faces added     : {log['faces_total']}
  Skipped         : {counters['skipped']}  (no face / unreadable)
  Failed          : {counters['failed']}
  Time            : {elapsed/60:.1f} minutes
  Qdrant total    : {final_info.points_count} faces
  Log             : {LOG_FILE}
{'='*55}
""")

    if counters['failed']:
        print("Failed files (re-run to retry):")
        for e in log["failed"][:10]:
            print(f"  {e.get('filename','?'):45s} {e.get('error','')[:50]}")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Live Event Mode (Drive Polling)
#
# Replaces the laptop file watcher entirely.
# Runs inside Colab — no laptop needed.
#
# What it does:
#   Every 30 seconds, checks Drive folder for new photos.
#   Any photo not in indexing_log.json gets indexed immediately.
#   Photographer just drags photos into the Drive folder —
#   they appear in guest search within ~35 seconds.
#
# Run AFTER Cell 7 finishes the initial batch index.
# Keep this cell running during the event.
# ════════════════════════════════════════════════════════════

import time, uuid, cv2, json, hashlib
import numpy as np
from pathlib import Path
from datetime import datetime
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

# ── Config ────────────────────────────────────────────────────
QDRANT_URL     = "your_qdrant_url_here"
QDRANT_API_KEY = "your_qdrant_api_key_here"
EVENT_ID       = "tantra_2026"
EMBEDDING_DIM  = 512

# Folder photographer uploads to
WATCH_DIR  = "/content/drive/MyDrive/TANTRA 2026/Assets_extracted"

# Same shared log from Cell 7
LOG_FILE   = "/content/drive/MyDrive/TANTRA 2026/indexing_log.json"

POLL_INTERVAL_SEC = 30
IMAGE_EXTS        = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


# ── Helpers ───────────────────────────────────────────────────

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def load_log():
    if Path(LOG_FILE).exists():
        try:
            with open(LOG_FILE) as f:
                return json.load(f)
        except Exception:
            pass
    return {"indexed": [], "skipped": [], "failed": [], "faces_total": 0}

def save_log(log):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)

def get_done_filenames(log) -> set:
    done = set()
    for entry in log["indexed"] + log["skipped"] + log["failed"]:
        if isinstance(entry, dict):
            name = entry.get("filename") or entry.get("file", "")
            if name:
                done.add(Path(name).name)
        elif isinstance(entry, str):
            done.add(Path(entry).name)
    return done

def index_photo(img_path: Path, log: dict) -> str:
    """Index one photo. Returns 'indexed' | 'skipped' | 'failed'."""
    try:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            log["skipped"].append({"filename": img_path.name, "reason": "unreadable"})
            return "skipped"

        h, w = img_bgr.shape[:2]
        if h * w > 4_000_000:
            scale   = (4_000_000 / (h * w)) ** 0.5
            img_bgr = cv2.resize(img_bgr, (int(w * scale), int(h * scale)))

        result = get_embeddings(img_bgr, augment=False)

        if not result["embeddings"]:
            log["skipped"].append({"filename": img_path.name, "reason": "no face"})
            return "skipped"

        points = [
            PointStruct(
                id=str(uuid.uuid4()),
                vector=emb,
                payload={
                    "image_path":    str(img_path),
                    "image_url":     "",
                    "filename":      img_path.name,
                    "media_type":    "image",
                    "source":        "drive_live",
                    "det_score":     1.0,
                    "quality_score": 1.0,
                    "event_id":      EVENT_ID,
                    "timestamp":     datetime.now().isoformat(),
                }
            )
            for emb in result["embeddings"]
        ]
        qdrant.upsert(collection_name=EVENT_ID, points=points)

        log["indexed"].append({
            "filename": img_path.name,
            "faces":    len(result["embeddings"]),
            "source":   "drive_live",
        })
        log["faces_total"] = log.get("faces_total", 0) + len(result["embeddings"])
        return "indexed"

    except Exception as e:
        log["failed"].append({"filename": img_path.name, "error": str(e)})
        return "failed"


# ── Main polling loop ─────────────────────────────────────────

print(f"Live event mode started.")
print(f"Watching: {WATCH_DIR}")
print(f"Polling every {POLL_INTERVAL_SEC} seconds.")
print(f"Press Stop (■) to end.\n")

session_indexed = 0
session_faces   = 0
poll_count      = 0

while True:
    poll_count += 1
    log  = load_log()
    done = get_done_filenames(log)

    # Find new photos
    try:
        watch_path = Path(WATCH_DIR)
        # rglob covers nested subfolders photographer might create
        all_photos = [
            p for p in watch_path.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTS
            and p.name not in done
        ]
    except Exception as e:
        print(f"  [Poll {poll_count}] Drive read error: {e}")
        time.sleep(POLL_INTERVAL_SEC)
        continue

    if all_photos:
        print(f"  [Poll {poll_count}] {len(all_photos)} new photo(s) found — indexing...")
        for photo in all_photos:
            t0     = time.time()
            result = index_photo(photo, log)
            elapsed = (time.time() - t0) * 1000

            if result == "indexed":
                faces = next(
                    (e["faces"] for e in reversed(log["indexed"])
                     if e.get("filename") == photo.name), 0
                )
                session_indexed += 1
                session_faces   += faces
                print(f"    ✅ {photo.name:45s} {faces} faces  {elapsed:.0f}ms")
            elif result == "skipped":
                print(f"    ⏭️  {photo.name:45s} no face detected")
            else:
                print(f"    ❌ {photo.name:45s} failed")

        save_log(log)

        qdrant_info = qdrant.get_collection(EVENT_ID)
        print(
            f"  Qdrant: {qdrant_info.points_count} total faces  "
            f"| Session: {session_indexed} photos, {session_faces} faces\n"
        )
    else:
        # Silent on no new photos — just show a dot every 5 polls
        if poll_count % 5 == 0:
            now = datetime.now().strftime("%H:%M:%S")
            print(f"  [{now}] Watching... "
                  f"(session: {session_indexed} new photos, "
                  f"{session_faces} faces indexed)")

    time.sleep(POLL_INTERVAL_SEC)